# AMEX Enterprise Credit Risk Platform
## Notebook 18 — Repository Packaging: GitHub, Kaggle & LinkedIn Portfolio Generation
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Deployment / Packaging**. Notebook 18 of 18 -- the final notebook. Depends on Notebook 01 only (the config file); every other notebook's real output is packaged opportunistically, whatever fraction of the platform has actually been run.

**What this notebook builds -- three separate, recruiter-facing packages, each following a distinct real-world convention:**

- A **GitHub Repository Package** (`data/`, `notebooks/`, `src/`, `models/`, `reports/`, `docs/`, `assets/`, `.gitignore`, `README.md`, `requirements.txt`, `LICENSE`) -- clean, professional, recruiter-friendly, with a comprehensive auto-generated README built entirely from this platform's real, live-computed results.
- A **Kaggle Project Package** (`data/`, `notebooks/` curated to Kaggle's simpler 5-notebook convention, `scripts/`, `models/`, `images/`, `README.md`, `requirements.txt`, `environment.yml`) -- with a transparent mapping table showing exactly which real platform notebook each Kaggle-style file corresponds to.
- A **LinkedIn Project Showcase** document (7-section structure: Overview, Data & Tools, Approach, Key Results, Takeaways, Future Work, Project Links) plus a suggested post caption -- real metrics wherever this platform has them, and clearly labeled **[EDITABLE]** placeholders for the personal narrative content (what you learned, challenges, links to your actual repos) that only you can honestly fill in.

**A hard size-safety rule enforced throughout:** GitHub rejects any pushed file over 100 MB and a bloated repo hurts the recruiter's first impression, so this notebook never copies raw competition data or large intermediate artifacts. Every single file placed into any package is checked against a live, configurable size cap *before* being copied -- if a real file exceeds the cap, or doesn't exist yet because an upstream notebook hasn't run, that is recorded honestly in a manifest, never silently dropped and never faked.

**Also included:** a real, live **code quality check** (syntax validation + optional static analysis via `pyflakes`) across every notebook and every standalone script this platform has generated, and a fresh **end-to-end pipeline flow diagram** for the documentation package.

**Deliverables:** `GitHub_Repository_Package/` (complete folder tree), `Kaggle_Project_Package/` (complete folder tree), `LinkedIn_Project_Showcase/` (docx + caption), `code_quality_report.csv`, `packaging_manifest.csv`, `platform_flow_diagram.png`, and `Repository_Packaging_Report.docx`.

**Run the single code cell below, once.** Idempotent — every package folder is fully cleared and rebuilt from scratch on every re-run, so it always reflects the platform's current real state.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOK 01 (SELF-HEALING)
# =============================================================================
import os
import sys
import json
import time
import shutil
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebook 01 (Self-Healing)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.\nFix: run 01_business_understanding.ipynb first -- "
                             f"this notebook reads its pillar directory map.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)

# --- Self-heal: a project_config.json written by an older copy of Notebook 01
#     (before Notebooks 17/18 existed) will not have these pillar keys yet.
#     Rather than require re-running Notebook 01 first, derive the standard
#     folder name, create it, and persist it back into project_config.json. ---
_REQUIRED_PILLARS = {"comprehensive_reporting": "Comprehensive_Reporting", "repository_packaging": "Repository_Packaging"}
_config_healed = False
for _key, _folder_name in _REQUIRED_PILLARS.items():
    if _key not in PILLAR_DIRS:
        PILLAR_DIRS[_key] = PROJECT_ROOT / _folder_name
        PROJECT_CONFIG["pillar_dirs"][_key] = str(PILLAR_DIRS[_key])
        _config_healed = True
        print(f"NOTE: '{_key}' was missing from project_config.json (an older Notebook 01 run) -- "
              f"added it automatically as {PILLAR_DIRS[_key]}")
if _config_healed:
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(PROJECT_CONFIG, f, indent=2)
    print(f"\u2705 project_config.json updated in place -- no need to re-run Notebook 01.")

REPO_PKG_DIR = PILLAR_DIRS["repository_packaging"]
REPO_PKG_DIR.mkdir(parents=True, exist_ok=True)

# --- Discover every notebook_0N_summary.json this run can find -- the same
#     generic scan used by Notebooks 15 and 17, so this final packaging
#     notebook stays honest about exactly how much of the platform has
#     actually been run. ---
NOTEBOOK_SUMMARIES = {}
for _p in sorted(ARTIFACTS_DIR.glob("notebook_*_summary.json")):
    try:
        _num = int(_p.stem.split("_")[1])
    except (IndexError, ValueError):
        continue
    with open(_p, "r", encoding="utf-8") as f:
        NOTEBOOK_SUMMARIES[_num] = json.load(f)

CORE_NOTEBOOK_NUMBERS = list(range(2, 17))
_core_found = [n for n in CORE_NOTEBOOK_NUMBERS if n in NOTEBOOK_SUMMARIES]
print(f"Notebook summaries found: {sorted(NOTEBOOK_SUMMARIES.keys())}")
print(f"Core pipeline coverage (Notebooks 02-16): {len(_core_found)} / {len(CORE_NOTEBOOK_NUMBERS)} found")
print(f"Real notebook files on disk (notebooks/): {len(list(NOTEBOOKS_DIR.glob('*.ipynb')))}")
print(f"Repository packaging will be written under: {REPO_PKG_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
    from docx.shared import Inches
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

import ast
import io
import textwrap

try:
    from pyflakes.api import check as _pyflakes_check
    from pyflakes.reporter import Reporter as _PyflakesReporter
    _HAS_PYFLAKES = True
except ImportError:
    _HAS_PYFLAKES = False

print(f"pyflakes available for static analysis: {_HAS_PYFLAKES}"
      + ("" if _HAS_PYFLAKES else "  (optional -- 'pip install pyflakes' for deeper static analysis; "
                                   "syntax-only checking will be used instead)"))
print("(Reporting only -- this notebook is I/O-bound packaging, not thread-parallelized work.)")


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
_live_vm = psutil.virtual_memory()
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(_live_vm.available * ADAPTIVE_RAM_FRACTION)
print(f"Adaptive RAM ceiling (this run) : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: PACKAGING SIZE-SAFETY POLICY & SAFE-COPY HELPER
# =============================================================================
_section("SECTION 3: Packaging Size-Safety Policy & Safe-Copy Helper")

# --- ASSUMPTION: these are deliberate packaging policy choices, not computed
#     platform facts -- editable. GitHub itself hard-blocks any pushed file
#     over 100 MB and recommends keeping a repository well under ~1 GB for a
#     fast clone; the caps below are set conservatively under that so a
#     recruiter's `git clone` stays fast. Kaggle's own dataset/notebook
#     limits are generous by comparison but this platform applies the same
#     size-safety discipline to every package it builds. Verify GitHub's
#     current, authoritative limits at https://docs.github.com before relying
#     on these numbers for a real push. ---
PACKAGING_POLICY = {
    "github_max_file_size_mb": 20.0,      # ASSUMPTION -- conservative, well under GitHub's 100 MB hard block
    "kaggle_max_file_size_mb": 50.0,       # ASSUMPTION -- Kaggle notebooks/datasets tolerate more than GitHub
    "linkedin_max_file_size_mb": 10.0,     # ASSUMPTION -- a LinkedIn post asset should be small regardless
    "copy_raw_competition_data": False,    # ASSUMPTION -- raw Kaggle CSVs are never redistributed; point to the source instead
}

PACKAGING_MANIFEST = []  # every copy decision this notebook makes, honestly recorded -- no silent drops


def _safe_copy(package: str, src_path: Path, dest_path: Path, max_mb: float, note: str = "") -> bool:
    """Copy a real file into a package only if it exists and is at or under
    the size cap for that package; otherwise records exactly why it was
    excluded. Never fabricates a placeholder file."""
    if not src_path.exists():
        PACKAGING_MANIFEST.append({"package": package, "source": str(src_path), "dest": str(dest_path),
                                    "included": False, "size_mb": None,
                                    "reason": "source file not found (upstream notebook has not run yet)"})
        return False
    _size_mb = round(src_path.stat().st_size / 1e6, 3)
    if _size_mb > max_mb:
        PACKAGING_MANIFEST.append({"package": package, "source": str(src_path), "dest": str(dest_path),
                                    "included": False, "size_mb": _size_mb,
                                    "reason": f"exceeds {package} packaging cap of {max_mb} MB"})
        return False
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_path, dest_path)
    PACKAGING_MANIFEST.append({"package": package, "source": str(src_path), "dest": str(dest_path),
                                "included": True, "size_mb": _size_mb, "reason": note or "copied"})
    return True


for _k, _v in PACKAGING_POLICY.items():
    print(f"  {_k:32s}: {_v}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LIVE CODE QUALITY CHECK -- EVERY REAL NOTEBOOK & STANDALONE SCRIPT
# =============================================================================
_section("SECTION 4: Live Code Quality Check -- Every Real Notebook & Standalone Script")

# --- A real, live static-analysis pass -- not a claimed pass/fail. Every
#     notebook this platform has actually produced (found on disk right now)
#     and every real standalone script (main.py, monitoring_job.py) is
#     syntax-checked with ast.parse(); if pyflakes is installed, a genuine
#     static-analysis warning count is added too. This never executes the
#     data pipeline -- only parses already-generated source text. ---

def _pyflakes_warning_count(source: str, filename: str):
    if not _HAS_PYFLAKES:
        return None
    _out, _err = io.StringIO(), io.StringIO()
    _reporter = _PyflakesReporter(_out, _err)
    try:
        return _pyflakes_check(source, filename, _reporter)
    except Exception:
        return None


def _check_source(label: str, source: str, kind: str) -> dict:
    _lines = source.count("\n") + 1
    try:
        _tree = ast.parse(source, filename=label)
        _syntax_valid = True
        _n_functions = sum(1 for n in ast.walk(_tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)))
        _note = ""
    except SyntaxError as exc:
        _syntax_valid = False
        _n_functions = 0
        _note = f"SyntaxError: {exc}"
    return {"file": label, "type": kind, "lines": _lines, "syntax_valid": _syntax_valid,
            "functions_defined": _n_functions,
            "pyflakes_warnings": _pyflakes_warning_count(source, label) if _syntax_valid else None,
            "notes": _note}


CODE_QUALITY_ROWS = []

# Every real notebook currently on disk, whatever fraction of the 18 that is.
for _nb_path in sorted(NOTEBOOKS_DIR.glob("*.ipynb")):
    try:
        with open(_nb_path, "r", encoding="utf-8") as f:
            _nb_json = json.load(f)
        _code_cells = [c for c in _nb_json.get("cells", []) if c.get("cell_type") == "code"]
        _source = "\n\n".join("".join(c.get("source", [])) for c in _code_cells)
        CODE_QUALITY_ROWS.append(_check_source(_nb_path.name, _source, "notebook"))
    except (json.JSONDecodeError, OSError) as exc:
        CODE_QUALITY_ROWS.append({"file": _nb_path.name, "type": "notebook", "lines": None,
                                   "syntax_valid": False, "functions_defined": None,
                                   "pyflakes_warnings": None, "notes": f"could not read/parse notebook: {exc}"})

# Every real standalone script this platform has actually generated.
_STANDALONE_SCRIPTS = [
    PILLAR_DIRS["fastapi_deployment"] / "main.py",
    PILLAR_DIRS["monitoring"] / "monitoring_job.py",
]
for _sp in _STANDALONE_SCRIPTS:
    if _sp.exists():
        with open(_sp, "r", encoding="utf-8") as f:
            CODE_QUALITY_ROWS.append(_check_source(_sp.name, f.read(), "script"))
    else:
        CODE_QUALITY_ROWS.append({"file": _sp.name, "type": "script", "lines": None, "syntax_valid": None,
                                   "functions_defined": None, "pyflakes_warnings": None,
                                   "notes": "not found -- upstream notebook has not run yet"})

code_quality_df = pd.DataFrame(CODE_QUALITY_ROWS)
_n_checked = len(code_quality_df)
_n_valid = int((code_quality_df["syntax_valid"] == True).sum())
code_quality_path = REPO_PKG_DIR / "code_quality_report.csv"
code_quality_df.to_csv(code_quality_path, index=False)
print(code_quality_df.to_string(index=False))
print(f"\nSyntax-valid: {_n_valid} / {_n_checked}")
print(f"\u2705 Saved -> {code_quality_path}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: SCAFFOLD THE THREE PACKAGE DIRECTORIES (FULL REBUILD EACH RUN)
# =============================================================================
_section("SECTION 5: Scaffold The Three Package Directories (Full Rebuild Each Run)")

# --- Unlike every other notebook's single overwritten output file, a
#     package is a whole directory tree -- true idempotency here means
#     clearing it completely before rebuilding, so a file that no longer
#     applies (e.g. a report from a notebook that hasn't run this time)
#     never lingers as a stale artifact from a previous run. ---
GITHUB_PKG_DIR = REPO_PKG_DIR / "GitHub_Repository_Package"
KAGGLE_PKG_DIR = REPO_PKG_DIR / "Kaggle_Project_Package"
LINKEDIN_PKG_DIR = REPO_PKG_DIR / "LinkedIn_Project_Showcase"

for _pkg_dir in (GITHUB_PKG_DIR, KAGGLE_PKG_DIR, LINKEDIN_PKG_DIR):
    if _pkg_dir.exists():
        shutil.rmtree(_pkg_dir)
    _pkg_dir.mkdir(parents=True, exist_ok=True)

print(f"\u2705 Cleared & re-created: {GITHUB_PKG_DIR}")
print(f"\u2705 Cleared & re-created: {KAGGLE_PKG_DIR}")
print(f"\u2705 Cleared & re-created: {LINKEDIN_PKG_DIR}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: "PROJECT AT A GLANCE" -- REAL HEADLINE METRICS FOR ALL THREE PACKAGES
# =============================================================================
_section("SECTION 6: Project At A Glance -- Real Headline Metrics")

# --- The same real-value-or-honest-fallback pattern used throughout this
#     platform -- every field below is read live from that notebook's own
#     summary artifact; a notebook that has not run contributes an honest
#     placeholder string, never a fabricated number. ---
def _g(n, *path, default="Not yet available -- run Notebook " ):
    d = NOTEBOOK_SUMMARIES.get(n)
    if d is None:
        return f"{default}{n:02d}"
    cur = d
    for p in path:
        if isinstance(cur, dict) and p in cur:
            cur = cur[p]
        else:
            return f"{default}{n:02d}"
    return cur


GLANCE = {
    "champion_model": _g(5, "champion_model"),
    "champion_holdout_amex_metric": _g(5, "champion_metrics", "holdout_amex_metric"),
    "champion_holdout_auc": _g(5, "champion_metrics", "holdout_auc"),
    "train_customers": _g(2, "train_data_csv", "customers_aggregated"),
    "test_customers": _g(2, "test_data_csv", "customers_aggregated"),
    "live_default_rate": _g(2, "join_validation", "live_default_rate"),
    "model_risk_tier": _g(7, "risk_tier"),
    "total_rwa_usd": _g(8, "total_rwa_usd"),
    "total_ecl_usd": _g(8, "total_ecl_usd"),
    "inference_p99_latency_ms": _g(9, "latency_summary", "single_row_p99_ms"),
    "batch_throughput_rows_per_sec": _g(9, "latency_summary", "batch_throughput_rows_per_sec"),
    "api_self_test_passed": _g(10, "api_self_test_passed"),
    "monitoring_alerts": _g(12, "n_alerts"),
    "base_5yr_roi_pct": _g(14, "base_5yr_roi_pct"),
    "base_5yr_net_benefit_usd": _g(14, "base_5yr_cumulative_net_benefit_usd"),
    "core_pipeline_completion_pct": _g(17, "core_pipeline_completion_pct"),
}
for _k, _v in GLANCE.items():
    print(f"  {_k:35s}: {_v}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: GITHUB PACKAGE -- FOLDER TREE, REAL FILE COPIES, .gitignore & LICENSE
# =============================================================================
_section("SECTION 7: GitHub Package -- Folder Tree, Real File Copies, .gitignore & LICENSE")

GH_CAP = PACKAGING_POLICY["github_max_file_size_mb"]

# --- notebooks/ -- every real notebook currently on disk ---
for _nb_path in sorted(NOTEBOOKS_DIR.glob("*.ipynb")):
    _safe_copy("github", _nb_path, GITHUB_PKG_DIR / "notebooks" / _nb_path.name, GH_CAP)

# --- src/ -- the real, standalone production code this platform generated ---
_safe_copy("github", PILLAR_DIRS["fastapi_deployment"] / "main.py", GITHUB_PKG_DIR / "src" / "fastapi_service" / "main.py", GH_CAP)
_safe_copy("github", PILLAR_DIRS["fastapi_deployment"] / "requirements-api.txt", GITHUB_PKG_DIR / "src" / "fastapi_service" / "requirements-api.txt", GH_CAP)
_safe_copy("github", PILLAR_DIRS["docker"] / "Dockerfile", GITHUB_PKG_DIR / "src" / "docker" / "Dockerfile", GH_CAP)
_safe_copy("github", PILLAR_DIRS["docker"] / "docker-compose.yml", GITHUB_PKG_DIR / "src" / "docker" / "docker-compose.yml", GH_CAP)
_safe_copy("github", PILLAR_DIRS["docker"] / ".dockerignore", GITHUB_PKG_DIR / "src" / "docker" / ".dockerignore", GH_CAP)
_safe_copy("github", PILLAR_DIRS["monitoring"] / "monitoring_job.py", GITHUB_PKG_DIR / "src" / "monitoring" / "monitoring_job.py", GH_CAP)

# --- reports/ -- every real Word report + its HTML companions, size-gated ---
_REPORT_SOURCES = [
    (PILLAR_DIRS["model_risk_management"] / "Model_Validation_Report.docx", "model_risk_management"),
    (PILLAR_DIRS["basel_ifrs9"] / "Basel_III_IFRS9_Mapping_Report.docx", "basel_ifrs9"),
    (PILLAR_DIRS["mlops"] / "MLOps_Model_Card_And_Deployment_Report.docx", "mlops"),
    (PILLAR_DIRS["fastapi_deployment"] / "FastAPI_Deployment_Report.docx", "fastapi_deployment"),
    (PILLAR_DIRS["docker"] / "Docker_Deployment_Report.docx", "docker"),
    (PILLAR_DIRS["monitoring"] / "Monitoring_Report.docx", "monitoring"),
    (PILLAR_DIRS["powerbi_dashboard"] / "PowerBI_Dashboard_Report.docx", "powerbi_dashboard"),
    (PILLAR_DIRS["powerbi_dashboard"] / "PowerBI_Dashboard_Preview.html", "powerbi_dashboard"),
    (PILLAR_DIRS["executive_reports"] / "Financial_Impact_Report.docx", "executive_reports"),
    (PILLAR_DIRS["executive_reports"] / "Financial_Impact_Dashboard.html", "executive_reports"),
    (PILLAR_DIRS["technical_documentation"] / "Technical_Documentation_Report.docx", "technical_documentation"),
    (PILLAR_DIRS["production_architecture"] / "Production_Architecture_Report.docx", "production_architecture"),
    (PILLAR_DIRS["comprehensive_reporting"] / "Comprehensive_Reporting_Report.docx", "comprehensive_reporting"),
]
for _src, _subdir in _REPORT_SOURCES:
    _safe_copy("github", _src, GITHUB_PKG_DIR / "reports" / _subdir / _src.name, GH_CAP)

# --- docs/ -- data dictionary, glossary, API reference, architecture map (all from Notebook 15) ---
for _fname in ["data_dictionary.csv", "glossary.csv", "api_reference.csv", "notebook_architecture_map.csv"]:
    _safe_copy("github", PILLAR_DIRS["technical_documentation"] / _fname, GITHUB_PKG_DIR / "docs" / _fname, GH_CAP)
_safe_copy("github", code_quality_path, GITHUB_PKG_DIR / "docs" / "code_quality_report.csv", GH_CAP)

# --- assets/ -- the key real diagrams and charts for embedding in README.md ---
_ASSET_SOURCES = [
    PILLAR_DIRS["technical_documentation"] / "architecture_diagram.png",
    PILLAR_DIRS["production_architecture"] / "production_architecture_diagram.png",
    PILLAR_DIRS["comprehensive_reporting"] / "platform_completion_chart.png",
    PILLAR_DIRS["comprehensive_reporting"] / "pillar_status_chart.png",
    PILLAR_DIRS["model_development"] / "model_comparison_chart.png",
    PILLAR_DIRS["explainable_ai"] / "shap_beeswarm_chart.png",
    PILLAR_DIRS["executive_reports"] / "financial_roi_by_horizon_chart.png",
    PILLAR_DIRS["powerbi_dashboard"] / "star_schema_diagram.png",
]
for _src in _ASSET_SOURCES:
    _safe_copy("github", _src, GITHUB_PKG_DIR / "assets" / _src.name, GH_CAP)

# --- models/ -- a documented registry, plus the real model files only if they fit the cap ---
_registry_path = PILLAR_DIRS["mlops"] / "model_registry.json"
_model_registry_entries = []
if _registry_path.exists():
    with open(_registry_path, "r", encoding="utf-8") as f:
        _model_registry_entries = json.load(f).get("entries", [])
_models_readme_lines = [
    "# Models", "",
    "Trained model binaries are intentionally **excluded from this GitHub package** by default "
    f"(the size-safety policy caps any single packaged file at {GH_CAP} MB) -- regenerate them "
    "by running `05_model_development.ipynb` and `09_mlops.ipynb` locally against the real Kaggle data.",
    "", "## Registered model versions (real, from Notebook 09's model registry)", "",
]
if _model_registry_entries:
    _models_readme_lines.append("| Model | Version | Registered (UTC) | Holdout AUC | Holdout AMEX Metric |")
    _models_readme_lines.append("|---|---|---|---|---|")
    for _e in _model_registry_entries:
        _models_readme_lines.append(
            f"| {_e.get('model_name', '')} | {_e.get('version', '')} | {_e.get('registered_at_utc', '')} | "
            f"{_e.get('holdout_auc', '')} | {_e.get('holdout_amex_metric', '')} |"
        )
else:
    _models_readme_lines.append("_Notebook 09 has not been run yet -- no registered model versions to list._")
(GITHUB_PKG_DIR / "models").mkdir(parents=True, exist_ok=True)
with open(GITHUB_PKG_DIR / "models" / "README.md", "w", encoding="utf-8") as f:
    f.write("\n".join(_models_readme_lines) + "\n")
_safe_copy("github", PILLAR_DIRS["model_development"] / "models" / "preprocessing_artifacts.joblib",
           GITHUB_PKG_DIR / "models" / "preprocessing_artifacts.joblib", GH_CAP,
           note="included only because it fit the size cap this run")

# --- data/ -- never the raw competition CSVs; a clear, honest pointer instead ---
_data_readme = [
    "# Data", "",
    "This platform is built on the real Kaggle **American Express Default Prediction** competition "
    "dataset. The raw CSVs (tens of gigabytes) are **not redistributed in this repository** -- "
    "download them yourself from the competition page and point `01_business_understanding.ipynb`'s "
    "`DATA_ROOT` at that folder to reproduce every result in this repository from scratch.",
    "", "Competition: https://www.kaggle.com/competitions/amex-default-prediction", "",
]
(GITHUB_PKG_DIR / "data").mkdir(parents=True, exist_ok=True)
with open(GITHUB_PKG_DIR / "data" / "README.md", "w", encoding="utf-8") as f:
    f.write("\n".join(_data_readme) + "\n")

# --- .gitignore ---
_gitignore_lines = [
    "# Python", "__pycache__/", "*.py[cod]", "*.egg-info/", ".Python", "build/", "dist/", "",
    "# Environments", ".env", ".venv", "venv/", "env/", "",
    "# Jupyter", ".ipynb_checkpoints/", "",
    "# OS", ".DS_Store", "Thumbs.db", "",
    "# Data & large artifacts -- never commit these, even if added locally later",
    "*.csv", "*.parquet", "*.joblib", "*.pkl", "data/raw/", "",
]
with open(GITHUB_PKG_DIR / ".gitignore", "w", encoding="utf-8") as f:
    f.write("\n".join(_gitignore_lines) + "\n")

# --- LICENSE (MIT, standard boilerplate) ---
_license_text = f"""MIT License

Copyright (c) {datetime.now().year} [Your Name]

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""
with open(GITHUB_PKG_DIR / "LICENSE", "w", encoding="utf-8") as f:
    f.write(_license_text)

# --- requirements.txt -- prefer Notebook 09's real, exported installed versions ---
_nb09_requirements = PILLAR_DIRS["mlops"] / "requirements.txt"
if _nb09_requirements.exists():
    _safe_copy("github", _nb09_requirements, GITHUB_PKG_DIR / "requirements.txt", GH_CAP,
               note="real, exported installed package versions from Notebook 09")
else:
    _fallback_requirements = [
        "polars", "numpy", "pandas", "scikit-learn", "xgboost", "lightgbm", "catboost",
        "shap", "lime", "matplotlib", "psutil", "joblib", "python-docx", "scipy",
        "openpyxl", "fastapi", "uvicorn", "pydantic", "pyyaml",
    ]
    with open(GITHUB_PKG_DIR / "requirements.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(_fallback_requirements) + "\n")
    PACKAGING_MANIFEST.append({"package": "github", "source": "(templated -- Notebook 09 has not run yet)",
                                "dest": str(GITHUB_PKG_DIR / "requirements.txt"), "included": True,
                                "size_mb": None, "reason": "documented baseline package list, not measured versions"})

print(f"\u2705 GitHub package populated under {GITHUB_PKG_DIR}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: GITHUB PACKAGE -- COMPREHENSIVE README.md (BUILT FROM REAL DATA)
# =============================================================================
_section("SECTION 8: GitHub Package -- Comprehensive README.md")


def _fmt_val(v, kind="num"):
    if isinstance(v, str):
        return v  # already an honest fallback string
    if kind == "pct" and isinstance(v, (int, float)):
        return f"{v:.2%}"
    if kind == "usd" and isinstance(v, (int, float)):
        return f"${v:,.0f}"
    if kind == "ms" and isinstance(v, (int, float)):
        return f"{v:.2f} ms"
    if kind == "int" and isinstance(v, (int, float)):
        return f"{v:,.0f}"
    return str(v)


def _render_tree(root: Path, max_entries: int = 14, _prefix: str = "") -> list:
    """A real, live-walked directory tree -- not a hand-typed diagram, so it
    can never drift from what this notebook actually built. Deep folders are
    capped per level with an honest '+N more' note, never silently dropped."""
    lines = []
    try:
        entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    except FileNotFoundError:
        return lines
    shown = entries[:max_entries]
    for i, entry in enumerate(shown):
        connector = "\u2514\u2500\u2500 " if i == len(shown) - 1 and len(entries) <= max_entries else "\u251c\u2500\u2500 "
        label = entry.name + ("/" if entry.is_dir() else "")
        lines.append(f"{_prefix}{connector}{label}")
        if entry.is_dir():
            extension = "    " if connector.startswith("\u2514") else "\u2502   "
            lines.extend(_render_tree(entry, max_entries, _prefix + extension))
    if len(entries) > max_entries:
        lines.append(f"{_prefix}\u2514\u2500\u2500 ... (+{len(entries) - max_entries} more)")
    return lines


_repo_tree_lines = [f"{GITHUB_PKG_DIR.name}/"] + _render_tree(GITHUB_PKG_DIR)

_readme_lines = [
    "# AMEX Enterprise Credit Risk Platform",
    "",
    "[![Python 3.11](https://img.shields.io/badge/python-3.11-blue.svg)]() "
    "[![CRISP-DM](https://img.shields.io/badge/methodology-CRISP--DM-informational.svg)]() "
    "[![License: MIT](https://img.shields.io/badge/license-MIT-green.svg)](LICENSE)",
    "",
    "An **18-notebook, enterprise-grade credit risk platform** built end-to-end on the real Kaggle "
    "**American Express Default Prediction** dataset -- data engineering, modeling, explainability, "
    "model risk management, Basel III / IFRS 9 regulatory mapping, MLOps, a live FastAPI scoring "
    "service, Docker containerization, production monitoring, a Power BI-ready data model, executive "
    "financial reporting, technical documentation, production architecture, and a comprehensive "
    "rollup report.",
    "",
    "## 1. Overview",
    "",
    f"- **Champion model (measured):** {_fmt_val(GLANCE['champion_model'])}",
    f"- **Champion holdout AMEX metric (measured):** {_fmt_val(GLANCE['champion_holdout_amex_metric'])}",
    f"- **Champion holdout AUC (measured):** {_fmt_val(GLANCE['champion_holdout_auc'])}",
    f"- **Core pipeline completion (Notebooks 02-16):** {_fmt_val(GLANCE['core_pipeline_completion_pct'])}%"
    if isinstance(GLANCE["core_pipeline_completion_pct"], (int, float)) else
    f"- **Core pipeline completion:** {GLANCE['core_pipeline_completion_pct']}",
    "",
    "## 2. Problem Statement",
    "",
    "Predict the probability that a credit card customer defaults, using the customer's real "
    "transaction and statement history, so a credit-risk team can price, provision, and manage "
    "portfolio risk proactively rather than reactively. Every displayed number in this repository "
    "is either computed live by that notebook's own code on the real dataset, or is an explicitly "
    "labeled, editable ASSUMPTION where the dataset genuinely has no ground truth for it (e.g. "
    "business financial assumptions) -- never silently presented as fact.",
    "",
    "## 3. Dataset",
    "",
    f"- Training customers aggregated (measured): {_fmt_val(GLANCE['train_customers'], 'int')}",
    f"- Test customers aggregated (measured): {_fmt_val(GLANCE['test_customers'], 'int')}",
    f"- Live default rate, join-validated (measured): {_fmt_val(GLANCE['live_default_rate'], 'pct')}",
    "- Source: [Kaggle -- American Express Default Prediction](https://www.kaggle.com/competitions/amex-default-prediction)",
    "  (raw CSVs are not redistributed here -- see `data/README.md`)",
    "",
    "## 4. Approach & Methodology",
    "",
    "Organized as 18 sequential notebooks around the CRISP-DM lifecycle: Business Understanding -> "
    "Data Engineering -> Data Validation/EDA -> Feature Engineering -> Model Development -> "
    "Explainable AI -> Model Risk Management -> Basel III/IFRS9 Mapping -> MLOps -> FastAPI "
    "Deployment -> Docker -> Monitoring -> Power BI Dashboard -> Executive Reports -> Technical "
    "Documentation -> Production Architecture -> Comprehensive Reporting -> Repository Packaging. "
    "See `docs/notebook_architecture_map.csv` for the full real dependency map, and "
    "`assets/architecture_diagram.png` below.",
    "",
    "![Architecture](assets/architecture_diagram.png)",
    "",
    "## 5. Key Results & Visuals",
    "",
    f"- Total RWA (computed, Basel III, Notebook 08): {_fmt_val(GLANCE['total_rwa_usd'], 'usd')}",
    f"- Total IFRS9 ECL (computed, Notebook 08): {_fmt_val(GLANCE['total_ecl_usd'], 'usd')}",
    f"- Inference p99 latency (measured, Notebook 09): {_fmt_val(GLANCE['inference_p99_latency_ms'], 'ms')}",
    f"- Batch throughput (measured, Notebook 09): {_fmt_val(GLANCE['batch_throughput_rows_per_sec'], 'int')} rows/sec",
    f"- Live API self-test (Notebook 10): {_fmt_val(GLANCE['api_self_test_passed'])}",
    f"- Base-scenario 5-year ROI (Notebook 14, ASSUMPTION-driven inputs): {_fmt_val(GLANCE['base_5yr_roi_pct'])}%"
    if isinstance(GLANCE["base_5yr_roi_pct"], (int, float)) else
    f"- Base-scenario 5-year ROI: {GLANCE['base_5yr_roi_pct']}",
    "",
    "![Model Comparison](assets/model_comparison_chart.png)",
    "![SHAP Beeswarm](assets/shap_beeswarm_chart.png)",
    "![Financial ROI by Horizon](assets/financial_roi_by_horizon_chart.png)",
    "",
    "## 6. How to Run",
    "",
    "1. Install Python 3.11 (Conda or venv) and the packages in `requirements.txt`.",
    "2. Download the real Kaggle competition CSVs (see `data/README.md`).",
    "3. Open `notebooks/01_business_understanding.ipynb`, set `DATA_ROOT`, run its single cell.",
    "4. Run the remaining notebooks in numeric order, 02 through 18 -- each is a single, idempotent "
    "code cell; every notebook's own intro states exactly which prior notebooks it depends on.",
    "",
    "## 7. Project Structure",
    "",
    "This tree is real -- generated by walking this exact package folder, not hand-typed:",
    "",
    "```",
] + _repo_tree_lines + [
    "```",
    "",
    "## 8. Technologies Used",
    "",
    "Polars (WARP-tuned thread pools), NumPy/Pandas, scikit-learn, XGBoost/LightGBM/CatBoost, "
    "SHAP & LIME, FastAPI + Uvicorn, Docker, Power BI (star schema + DAX), python-docx/Matplotlib "
    "for reporting, psutil for adaptive resource management.",
    "",
    "## 9. Code Quality",
    "",
    f"A live syntax and static-analysis pass runs across every notebook and standalone script this "
    f"platform generates -- see `docs/code_quality_report.csv`. {_n_valid} / {_n_checked} files "
    f"passed a syntax check as of this run"
    + (" (static analysis via pyflakes included)." if _HAS_PYFLAKES else " (pyflakes not installed -- syntax-only)."),
    "",
    "## 10. Future Work",
    "",
    "See `Production_Architecture_Report.docx` (in `reports/production_architecture/`) for every "
    "**RECOMMENDED** (not yet built) component this platform's own architecture review identified -- "
    "for example a managed feature store, an orchestration layer (Airflow/Prefect), and a real CI/CD "
    "pipeline for the FastAPI service.",
    "",
    "## License",
    "",
    "MIT -- see `LICENSE`.",
    "",
    f"_Generated by 18_repository_packaging.ipynb, {datetime.now().strftime('%Y-%m-%d %H:%M')}._",
    "",
]
GITHUB_README_CONTENT = "\n".join(_readme_lines)
github_readme_path = GITHUB_PKG_DIR / "README.md"
with open(github_readme_path, "w", encoding="utf-8") as f:
    f.write(GITHUB_README_CONTENT)
print(f"\u2705 Saved -> {github_readme_path}  ({len(GITHUB_README_CONTENT):,} characters)")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: KAGGLE PACKAGE -- CURATED NOTEBOOK MAPPING, SCRIPTS, MODELS & IMAGES
# =============================================================================
_section("SECTION 9: Kaggle Package -- Curated Notebook Mapping, Scripts, Models & Images")

KG_CAP = PACKAGING_POLICY["kaggle_max_file_size_mb"]

# --- Kaggle's convention is a simpler 5-notebook flow. This platform's real
#     18 notebooks are richer than that, so this section curates a transparent
#     mapping -- real notebooks, renamed to Kaggle's convention -- documented
#     explicitly so nothing is silently relabeled. ---
KAGGLE_NOTEBOOK_MAPPING = [
    {"kaggle_name": "01_eda.ipynb", "source_name": "03_data_validation_eda.ipynb", "kaggle_label": "Exploratory Data Analysis"},
    {"kaggle_name": "02_feature_engineering.ipynb", "source_name": "04_feature_engineering.ipynb", "kaggle_label": "Feature Engineering"},
    {"kaggle_name": "03_modeling.ipynb", "source_name": "05_model_development.ipynb", "kaggle_label": "Model Building"},
    {"kaggle_name": "04_evaluation.ipynb", "source_name": "06_explainable_ai.ipynb", "kaggle_label": "Model Evaluation (Explainability)"},
    {"kaggle_name": "05_visualizations.ipynb", "source_name": "14_executive_reports.ipynb", "kaggle_label": "Results & Insights"},
]
for _m in KAGGLE_NOTEBOOK_MAPPING:
    _src = NOTEBOOKS_DIR / _m["source_name"]
    _ok = _safe_copy("kaggle", _src, KAGGLE_PKG_DIR / "notebooks" / _m["kaggle_name"], KG_CAP,
                      note=f"renamed from real {_m['source_name']}")
    _m["included"] = _ok

_mapping_lines = ["# Notebook Mapping", "",
                   "This package follows Kaggle's simpler 5-notebook convention. Each file below is a "
                   "**real, unmodified copy** of one of this platform's actual 18 notebooks, renamed for "
                   "Kaggle's structure -- nothing here is a rewritten or fabricated summary.", "",
                   "| Kaggle file | Real source notebook | Purpose |", "|---|---|---|"]
for _m in KAGGLE_NOTEBOOK_MAPPING:
    _status = "" if _m["included"] else "  _(not yet run -- excluded this build)_"
    _mapping_lines.append(f"| `{_m['kaggle_name']}` | `{_m['source_name']}` | {_m['kaggle_label']}{_status} |")
(KAGGLE_PKG_DIR / "notebooks").mkdir(parents=True, exist_ok=True)
with open(KAGGLE_PKG_DIR / "notebooks" / "NOTEBOOK_MAPPING.md", "w", encoding="utf-8") as f:
    f.write("\n".join(_mapping_lines) + "\n")

# --- scripts/ -- only the real standalone scripts this platform produces;
#     preprocessing/training live inside the notebooks themselves by this
#     platform's own convention, and that is stated honestly rather than
#     inventing placeholder scripts for slots that have no real equivalent. ---
_safe_copy("kaggle", PILLAR_DIRS["fastapi_deployment"] / "main.py", KAGGLE_PKG_DIR / "scripts" / "predict_service.py", KG_CAP,
           note="real FastAPI serving/inference code from Notebook 10")
_safe_copy("kaggle", PILLAR_DIRS["monitoring"] / "monitoring_job.py", KAGGLE_PKG_DIR / "scripts" / "monitoring_job.py", KG_CAP,
           note="real scheduled monitoring script from Notebook 12")
_scripts_readme = [
    "# Scripts", "",
    "This platform's data preprocessing, feature engineering, and model training are implemented as "
    "single consolidated notebook cells (see `notebooks/02_feature_engineering.ipynb` and "
    "`notebooks/03_modeling.ipynb`) rather than standalone `.py` scripts -- a deliberate platform "
    "convention, not an omission. The two real standalone scripts this platform *does* produce are "
    "included here: `predict_service.py` (the live FastAPI inference service) and `monitoring_job.py` "
    "(the scheduled drift-monitoring job).", "",
]
(KAGGLE_PKG_DIR / "scripts").mkdir(parents=True, exist_ok=True)
with open(KAGGLE_PKG_DIR / "scripts" / "README.md", "w", encoding="utf-8") as f:
    f.write("\n".join(_scripts_readme) + "\n")

# --- models/ -- same registry documentation as the GitHub package ---
(KAGGLE_PKG_DIR / "models").mkdir(parents=True, exist_ok=True)
with open(KAGGLE_PKG_DIR / "models" / "README.md", "w", encoding="utf-8") as f:
    f.write("\n".join(_models_readme_lines) + "\n")
_safe_copy("kaggle", PILLAR_DIRS["model_development"] / "models" / "preprocessing_artifacts.joblib",
           KAGGLE_PKG_DIR / "models" / "preprocessing_artifacts.joblib", KG_CAP)

# --- images/eda_plots/ and images/results/ -- a curated, real selection ---
_EDA_CHARTS = ["01_target_distribution.png", "02_missing_value_profile.png",
               "03_feature_distributions.png", "04_correlation_heatmap.png"]
for _c in _EDA_CHARTS:
    _safe_copy("kaggle", PILLAR_DIRS["data_validation"] / _c, KAGGLE_PKG_DIR / "images" / "eda_plots" / _c, KG_CAP)

_RESULT_CHARTS = [
    (PILLAR_DIRS["model_development"], "model_comparison_chart.png"),
    (PILLAR_DIRS["explainable_ai"], "shap_beeswarm_chart.png"),
    (PILLAR_DIRS["executive_reports"], "financial_roi_by_horizon_chart.png"),
    (PILLAR_DIRS["comprehensive_reporting"], "platform_completion_chart.png"),
]
for _dir, _c in _RESULT_CHARTS:
    _safe_copy("kaggle", _dir / _c, KAGGLE_PKG_DIR / "images" / "results" / _c, KG_CAP)

# --- data/input, data/output ---
(KAGGLE_PKG_DIR / "data" / "input").mkdir(parents=True, exist_ok=True)
with open(KAGGLE_PKG_DIR / "data" / "input" / "README.md", "w", encoding="utf-8") as f:
    f.write("Original Kaggle competition data -- not redistributed here. See the main README.\n")
for _fname in ["model_comparison.csv", "champion_feature_importance.csv"]:
    _safe_copy("kaggle", PILLAR_DIRS["model_development"] / _fname, KAGGLE_PKG_DIR / "data" / "output" / _fname, KG_CAP)

print(f"\u2705 Kaggle package populated under {KAGGLE_PKG_DIR}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: KAGGLE PACKAGE -- README.md, requirements.txt & environment.yml
# =============================================================================
_section("SECTION 10: Kaggle Package -- README.md, requirements.txt & environment.yml")

_kaggle_readme_lines = [
    "# AMEX Default Prediction -- Credit Risk Scoring (Kaggle Project Package)",
    "",
    "## Overview",
    "",
    f"Champion model (measured): **{_fmt_val(GLANCE['champion_model'])}** -- holdout AMEX metric "
    f"**{_fmt_val(GLANCE['champion_holdout_amex_metric'])}**, holdout AUC **{_fmt_val(GLANCE['champion_holdout_auc'])}**.",
    "", "## Dataset", "",
    "Kaggle competition: American Express Default Prediction "
    "(https://www.kaggle.com/competitions/amex-default-prediction). Raw data is not redistributed "
    "here -- see `data/input/README.md`.",
    "", "## Methodology", "",
    "See `notebooks/NOTEBOOK_MAPPING.md` for exactly which real, full platform notebook each file "
    "here is drawn from -- this package is a curated subset of an 18-notebook enterprise platform, "
    "not a from-scratch rewrite.",
    "", "## Results", "",
    f"- Total RWA (computed): {_fmt_val(GLANCE['total_rwa_usd'], 'usd')}",
    f"- Total IFRS9 ECL (computed): {_fmt_val(GLANCE['total_ecl_usd'], 'usd')}",
    f"- Inference p99 latency (measured): {_fmt_val(GLANCE['inference_p99_latency_ms'], 'ms')}",
    "", "## How to Run", "",
    "1. Install dependencies from `requirements.txt` (or `environment.yml` for conda).",
    "2. Run the notebooks in `notebooks/` in the order shown in `NOTEBOOK_MAPPING.md`.",
    "", "## Dependencies", "",
    "See `requirements.txt` / `environment.yml`.",
    "", "## Acknowledgements", "",
    "Built on the Kaggle American Express Default Prediction competition dataset, provided by "
    "American Express via Kaggle.",
    "", f"_Generated by 18_repository_packaging.ipynb, {datetime.now().strftime('%Y-%m-%d %H:%M')}._", "",
]
with open(KAGGLE_PKG_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write("\n".join(_kaggle_readme_lines) + "\n")

if _nb09_requirements.exists():
    _safe_copy("kaggle", _nb09_requirements, KAGGLE_PKG_DIR / "requirements.txt", KG_CAP)
else:
    with open(KAGGLE_PKG_DIR / "requirements.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(_fallback_requirements) + "\n")

_environment_yml_lines = [
    "name: amex-credit-risk", "channels:", "  - conda-forge", "  - defaults", "dependencies:",
    "  - python=3.11", "  - pip", "  - pip:",
] + [f"      - {pkg}" for pkg in ["polars", "numpy", "pandas", "scikit-learn", "xgboost", "lightgbm",
                                    "catboost", "shap", "lime", "matplotlib", "psutil", "joblib",
                                    "python-docx", "scipy", "openpyxl", "fastapi", "uvicorn", "pydantic", "pyyaml"]]
with open(KAGGLE_PKG_DIR / "environment.yml", "w", encoding="utf-8") as f:
    f.write("\n".join(_environment_yml_lines) + "\n")

print(f"\u2705 Saved -> {KAGGLE_PKG_DIR / 'README.md'}")
print(f"\u2705 Saved -> {KAGGLE_PKG_DIR / 'requirements.txt'}")
print(f"\u2705 Saved -> {KAGGLE_PKG_DIR / 'environment.yml'}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: LINKEDIN PROJECT SHOWCASE -- 7-SECTION DOCX + SUGGESTED CAPTION
# =============================================================================
_section("SECTION 11: LinkedIn Project Showcase -- 7-Section Docx + Suggested Caption")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_bullets(doc, items):
    for it in items:
        doc.add_paragraph(it, style="List Bullet")


li = Document()
li.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
li.add_paragraph("LinkedIn Project Showcase")
li.add_paragraph(
    "Every metric below is real, pulled live from this platform's own notebooks. Sections marked "
    "[EDITABLE] are personal narrative content only you can honestly write -- fill them in before "
    "posting; nothing has been invented on your behalf."
)

_add_heading(li, "1. Project Overview", level=1)
li.add_paragraph("Project Title: AMEX Enterprise Credit Risk Platform")
li.add_paragraph(
    "One-line description: An 18-notebook, enterprise-grade credit risk platform -- data engineering "
    "through model development, explainability, regulatory capital mapping, MLOps, live deployment, "
    "monitoring, BI, and executive financial reporting -- built end-to-end on the real Kaggle American "
    "Express Default Prediction dataset."
)
li.add_paragraph(
    "Problem statement & objective: Predict customer default probability from real transaction and "
    "statement history to support proactive credit-risk pricing, provisioning, and portfolio "
    "management."
)

_add_heading(li, "2. Data & Tools", level=1)
_add_bullets(li, [
    f"Data source & size (measured): {_fmt_val(GLANCE['train_customers'], 'int')} training customers, "
    f"{_fmt_val(GLANCE['test_customers'], 'int')} test customers, from the Kaggle AMEX Default "
    f"Prediction competition.",
    "Key tools & technologies: Polars, XGBoost/LightGBM/CatBoost, scikit-learn, SHAP/LIME, FastAPI, "
    "Docker, Power BI.",
    "Programming language: Python 3.11.",
])

_add_heading(li, "3. Approach & Methodology", level=1)
_add_bullets(li, [
    "Step-by-step approach: 18 sequential notebooks organized around the CRISP-DM lifecycle, from "
    "business understanding through deployment and monitoring.",
    f"Models & techniques used (measured): champion model {_fmt_val(GLANCE['champion_model'])}, "
    "selected by holdout AMEX metric across a real multi-model comparison; SHAP and LIME for "
    "explainability; PSI-based drift monitoring in production.",
    "Why this approach: every displayed number is either computed live on the real dataset or an "
    "explicitly labeled, editable assumption -- built to survive real scrutiny, not just look good "
    "in a demo.",
])

_add_heading(li, "4. Key Results & Impact", level=1)
_add_bullets(li, [
    f"Champion holdout AMEX metric (measured): {_fmt_val(GLANCE['champion_holdout_amex_metric'])}",
    f"Champion holdout AUC (measured): {_fmt_val(GLANCE['champion_holdout_auc'])}",
    f"Total RWA / IFRS9 ECL (computed): {_fmt_val(GLANCE['total_rwa_usd'], 'usd')} / "
    f"{_fmt_val(GLANCE['total_ecl_usd'], 'usd')}",
    f"Inference p99 latency (measured): {_fmt_val(GLANCE['inference_p99_latency_ms'], 'ms')}",
    f"Base-scenario 5-year ROI (ASSUMPTION-driven business inputs): "
    + (f"{GLANCE['base_5yr_roi_pct']}%" if isinstance(GLANCE["base_5yr_roi_pct"], (int, float)) else str(GLANCE["base_5yr_roi_pct"])),
])

_add_heading(li, "5. Key Takeaways [EDITABLE -- personalize before posting]", level=1)
_add_bullets(li, [
    "[EDITABLE] What I learned: ...",
    "[EDITABLE] Challenges & how I solved them: ...",
    "[EDITABLE] Skills & growth: ...",
])

_add_heading(li, "6. Future Work", level=1)
li.add_paragraph(
    "See the Production Architecture report's RECOMMENDED (not yet built) components for real, "
    "platform-identified next steps -- e.g. a managed feature store, workflow orchestration, and a "
    "CI/CD pipeline for the FastAPI service."
)
li.add_paragraph("[EDITABLE] Additional next steps / broader applications you'd highlight: ...")

_add_heading(li, "7. Project Links [EDITABLE -- add your real links before posting]", level=1)
_add_bullets(li, [
    "[EDITABLE] GitHub Repository Link: ...",
    "[EDITABLE] Kaggle Notebook / Dataset Link: ...",
    "[EDITABLE] Dashboard / Demo Link (if any): ...",
])

linkedin_docx_path = LINKEDIN_PKG_DIR / "LinkedIn_Project_Showcase.docx"
li.save(str(linkedin_docx_path))

_caption_lines = [
    "[EDITABLE -- suggested draft, personalize before posting]", "",
    "Just wrapped up a full end-to-end credit risk platform on the real Kaggle American Express "
    "Default Prediction dataset -- 18 notebooks covering data engineering, modeling, explainability, "
    "Basel III / IFRS9 regulatory mapping, MLOps, live FastAPI deployment, monitoring, and Power BI "
    "reporting.", "",
    f"Champion model: {_fmt_val(GLANCE['champion_model'])} | Holdout AMEX metric: "
    f"{_fmt_val(GLANCE['champion_holdout_amex_metric'])}", "",
    "#MachineLearning #CreditRisk #DataScience #FinTech #MLOps #Python #Kaggle",
]
caption_path = LINKEDIN_PKG_DIR / "linkedin_post_caption.txt"
with open(caption_path, "w", encoding="utf-8") as f:
    f.write("\n".join(_caption_lines) + "\n")

_cover_src = PILLAR_DIRS["comprehensive_reporting"] / "platform_completion_chart.png"
if not _cover_src.exists():
    _cover_src = PILLAR_DIRS["technical_documentation"] / "architecture_diagram.png"
_safe_copy("linkedin", _cover_src, LINKEDIN_PKG_DIR / "suggested_cover_image.png",
           PACKAGING_POLICY["linkedin_max_file_size_mb"])

print(f"\u2705 Saved -> {linkedin_docx_path}")
print(f"\u2705 Saved -> {caption_path}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: FRESH END-TO-END PIPELINE FLOW DIAGRAM (FOR docs/ IN BOTH PACKAGES)
# =============================================================================
_section("SECTION 12: Fresh End-to-End Pipeline Flow Diagram")

PROBLEM_NAME = "Phase 1 \u00b7 Problem 1 -- Credit Scoring / PD Prediction"
VIZ = {"surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
       "cat_blue": "#2a78d6", "cat_green": "#3a9e5f", "cat_grey": "#9c9b96"}

FLOW_STAGES = [
    {"label": "Business &\nData (01-04)", "notebooks": [1, 2, 3, 4]},
    {"label": "Modeling &\nExplainability (05-06)", "notebooks": [5, 6]},
    {"label": "Risk &\nRegulatory (07-08)", "notebooks": [7, 8]},
    {"label": "Deployment &\nServing (09-11)", "notebooks": [9, 10, 11]},
    {"label": "Monitoring &\nBI (12-13)", "notebooks": [12, 13]},
    {"label": "Reporting &\nPackaging (14-18)", "notebooks": [14, 15, 16, 17, 18]},
]


def _stage_has_run(nbs):
    return all((n in NOTEBOOK_SUMMARIES) or (n in (1, 17, 18)) for n in nbs)


fig, ax = plt.subplots(figsize=(13, 3.6), dpi=150)
ax.set_facecolor(VIZ["surface"]); fig.set_facecolor(VIZ["surface"])
ax.axis("off")
_n_stages = len(FLOW_STAGES)
_box_w, _gap = 1.8, 0.55
for i, stage in enumerate(FLOW_STAGES):
    _x = i * (_box_w + _gap)
    _color = VIZ["cat_green"] if _stage_has_run(stage["notebooks"]) else VIZ["cat_grey"]
    ax.add_patch(plt.Rectangle((_x, 0), _box_w, 1.4, facecolor=_color, edgecolor="white", linewidth=2))
    ax.text(_x + _box_w / 2, 0.7, stage["label"], ha="center", va="center", fontsize=8.5, color="white", weight="bold")
    if i < _n_stages - 1:
        ax.annotate("", xy=(_x + _box_w + _gap - 0.05, 0.7), xytext=(_x + _box_w + 0.05, 0.7),
                    arrowprops=dict(arrowstyle="->", color=VIZ["text_secondary"], lw=1.6))
ax.set_xlim(-0.2, _n_stages * (_box_w + _gap))
ax.set_ylim(-0.3, 1.9)
_legend_patches = [plt.Rectangle((0, 0), 1, 1, facecolor=VIZ["cat_green"], label="Complete"),
                    plt.Rectangle((0, 0), 1, 1, facecolor=VIZ["cat_grey"], label="Incomplete")]
ax.legend(handles=_legend_patches, loc="upper center", bbox_to_anchor=(0.5, -0.05), fontsize=8, frameon=False, ncol=2)
ax.set_title(f"{PROBLEM_NAME}\nEnd-to-End Pipeline Flow (Live Run Status)", fontsize=11, color=VIZ["text_primary"])
fig.tight_layout()
flow_diagram_path = REPO_PKG_DIR / "platform_flow_diagram.png"
fig.savefig(flow_diagram_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)

for _pkg_dir in (GITHUB_PKG_DIR, KAGGLE_PKG_DIR):
    _dest = _pkg_dir / "docs" / "platform_flow_diagram.png"
    _dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(flow_diagram_path, _dest)

print(f"\u2705 Saved -> {flow_diagram_path}")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: PACKAGE MANIFESTS & SIZE REPORTS
# =============================================================================
_section("SECTION 13: Package Manifests & Size Reports")

manifest_df = pd.DataFrame(PACKAGING_MANIFEST)
manifest_path = REPO_PKG_DIR / "packaging_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)


def _dir_size_mb(root: Path) -> float:
    if not root.exists():
        return 0.0
    return round(sum(f.stat().st_size for f in root.rglob("*") if f.is_file()) / 1e6, 2)


def _dir_file_count(root: Path) -> int:
    if not root.exists():
        return 0
    return sum(1 for f in root.rglob("*") if f.is_file())


PACKAGE_SIZE_REPORT = pd.DataFrame([
    {"package": "GitHub_Repository_Package", "files": _dir_file_count(GITHUB_PKG_DIR),
     "total_size_mb": _dir_size_mb(GITHUB_PKG_DIR), "per_file_cap_mb": GH_CAP},
    {"package": "Kaggle_Project_Package", "files": _dir_file_count(KAGGLE_PKG_DIR),
     "total_size_mb": _dir_size_mb(KAGGLE_PKG_DIR), "per_file_cap_mb": KG_CAP},
    {"package": "LinkedIn_Project_Showcase", "files": _dir_file_count(LINKEDIN_PKG_DIR),
     "total_size_mb": _dir_size_mb(LINKEDIN_PKG_DIR), "per_file_cap_mb": PACKAGING_POLICY["linkedin_max_file_size_mb"]},
])
package_size_report_path = REPO_PKG_DIR / "package_size_report.csv"
PACKAGE_SIZE_REPORT.to_csv(package_size_report_path, index=False)

_n_included = int(manifest_df["included"].sum()) if len(manifest_df) else 0
_n_excluded = int((~manifest_df["included"]).sum()) if len(manifest_df) else 0
print(manifest_df.to_string(index=False) if len(manifest_df) <= 60 else manifest_df.tail(60).to_string(index=False))
print(f"\n{PACKAGE_SIZE_REPORT.to_string(index=False)}")
print(f"\nFiles included across all packages: {_n_included}  |  Excluded (missing or over cap): {_n_excluded}")
print(f"\u2705 Saved -> {manifest_path}")
print(f"\u2705 Saved -> {package_size_report_path}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: REPOSITORY PACKAGING READINESS CHECKLIST
# =============================================================================
_section("SECTION 14: Repository Packaging Readiness Checklist")

packaging_checklist = [
    {"dimension": "Code Quality Check Ran (Live, Every Real File)", "status": "Pass",
     "evidence": f"{_n_valid} / {_n_checked} files syntax-valid"},
    {"dimension": "GitHub Package Built & Size-Gated", "status": "Pass",
     "evidence": f"{_dir_file_count(GITHUB_PKG_DIR)} files, {_dir_size_mb(GITHUB_PKG_DIR)} MB"},
    {"dimension": "Kaggle Package Built & Size-Gated", "status": "Pass",
     "evidence": f"{_dir_file_count(KAGGLE_PKG_DIR)} files, {_dir_size_mb(KAGGLE_PKG_DIR)} MB"},
    {"dimension": "LinkedIn Showcase Built", "status": "Pass",
     "evidence": f"{_dir_file_count(LINKEDIN_PKG_DIR)} files, {_dir_size_mb(LINKEDIN_PKG_DIR)} MB"},
    {"dimension": "No Raw Competition Data Redistributed", "status": "Pass" if not PACKAGING_POLICY["copy_raw_competition_data"] else "FAIL",
     "evidence": "policy: copy_raw_competition_data=False"},
    {"dimension": "Every Packaging Decision Logged (No Silent Drops)", "status": "Pass",
     "evidence": f"{len(manifest_df)} manifest rows"},
    {"dimension": "Kaggle Notebook Mapping Documented", "status": "Pass",
     "evidence": f"{sum(1 for m in KAGGLE_NOTEBOOK_MAPPING if m['included'])} / {len(KAGGLE_NOTEBOOK_MAPPING)} mapped notebooks included"},
]
packaging_checklist_df = pd.DataFrame(packaging_checklist)
packaging_checklist_path = REPO_PKG_DIR / "repository_packaging_checklist.csv"
packaging_checklist_df.to_csv(packaging_checklist_path, index=False)
print(packaging_checklist_df.to_string(index=False))
print(f"\u2705 Saved -> {packaging_checklist_path}")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: WORD REPORT -- REPOSITORY_PACKAGING_REPORT.DOCX
# =============================================================================
_section("SECTION 15: Word Report -- Repository_Packaging_Report.docx")


def _add_table_from_df(doc, df, max_rows=40):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    if len(df) > max_rows:
        doc.add_paragraph(f"... and {len(df) - max_rows} more row(s) -- see the full CSV for the complete table.")
    return table


report = Document()
report.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
report.add_paragraph("Repository Packaging Report -- Notebook 18 (Final)")
report.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(report, "1. Package Size Report", level=1)
report.add_paragraph(
    "Every file in every package was checked against a live size cap before being copied -- GitHub "
    "blocks any pushed file over 100 MB, so this platform caps well below that by policy."
)
_add_table_from_df(report, PACKAGE_SIZE_REPORT)

_add_heading(report, "2. End-to-End Pipeline Flow", level=1)
report.add_picture(str(flow_diagram_path), width=Inches(6.3))

_add_heading(report, "3. GitHub Repository Package", level=1)
report.add_paragraph(f"Built under {GITHUB_PKG_DIR}")
for _line in _repo_tree_lines[:30]:
    report.add_paragraph(_line, style="No Spacing")

_add_heading(report, "4. Kaggle Project Package -- Notebook Mapping", level=1)
_add_table_from_df(report, pd.DataFrame(KAGGLE_NOTEBOOK_MAPPING))

_add_heading(report, "5. LinkedIn Project Showcase", level=1)
report.add_paragraph(
    "A 7-section showcase document was generated at "
    f"{linkedin_docx_path.relative_to(REPO_PKG_DIR)} following the Project Overview / Data & Tools / "
    "Approach & Methodology / Key Results & Impact / Key Takeaways / Future Work / Project Links "
    "structure. Sections requiring personal narrative are clearly marked [EDITABLE]."
)

_add_heading(report, "6. Code Quality Report", level=1)
_add_table_from_df(report, code_quality_df)

_add_heading(report, "7. Packaging Manifest (What Was Included / Excluded, and Why)", level=1)
report.add_paragraph(
    "Paths below are shown relative to the project root for readability; the full paths are in "
    "packaging_manifest.csv."
)
_manifest_display_df = manifest_df.copy()
for _col in ("source", "dest"):
    _manifest_display_df[_col] = _manifest_display_df[_col].apply(
        lambda p: str(Path(p).relative_to(PROJECT_ROOT)) if p and str(PROJECT_ROOT) in str(p) else p
    )
_add_table_from_df(report, _manifest_display_df, max_rows=50)

_add_heading(report, "8. Repository Packaging Readiness Checklist", level=1)
_add_table_from_df(report, packaging_checklist_df)

report_path = REPO_PKG_DIR / "Repository_Packaging_Report.docx"
report.save(str(report_path))
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 15 complete.")


# =============================================================================
# SECTION 16: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 16: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("GitHub package has a README.md", (GITHUB_PKG_DIR / "README.md").exists())
_check("GitHub package has a LICENSE", (GITHUB_PKG_DIR / "LICENSE").exists())
_check("GitHub package has a .gitignore", (GITHUB_PKG_DIR / ".gitignore").exists())
_check("Kaggle package has a README.md", (KAGGLE_PKG_DIR / "README.md").exists())
_check("Kaggle package has an environment.yml", (KAGGLE_PKG_DIR / "environment.yml").exists())
_check("LinkedIn showcase docx exists", linkedin_docx_path.exists())
_check("No file in any package exceeds its package's size cap",
       manifest_df.loc[manifest_df["included"], "size_mb"].fillna(0).astype(float).max()
       <= max(GH_CAP, KG_CAP, PACKAGING_POLICY["linkedin_max_file_size_mb"]) if manifest_df["included"].any() else True)
_check("Raw competition CSVs were never copied into any package",
       not any(("train_data.csv" in str(m.get("dest", "")) or "test_data.csv" in str(m.get("dest", "")))
               and m.get("included") for m in PACKAGING_MANIFEST))
_check("Packaging manifest is non-empty", len(manifest_df) > 0)
_check("Code quality report covers every real notebook on disk",
       len(code_quality_df[code_quality_df["type"] == "notebook"]) == len(list(NOTEBOOKS_DIR.glob("*.ipynb"))))

_expected_files = [code_quality_path, manifest_path, package_size_report_path, packaging_checklist_path,
                    flow_diagram_path, github_readme_path, linkedin_docx_path, caption_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 18 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 18 checks passed.")
print("\n\u2705 Section 16 complete.")


# =============================================================================
# SECTION 17: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 17: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
    "pyflakes_available": _HAS_PYFLAKES,
}
performance_report_path = ARTIFACTS_DIR / "notebook_18_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 17 complete.")


# =============================================================================
# SECTION 18: WRITE NOTEBOOK 18 SUMMARY ARTIFACT
# =============================================================================
_section("SECTION 18: Write Notebook 18 Summary Artifact")

notebook_18_summary = {
    "notebook": "18_repository_packaging", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "github_package_files": _dir_file_count(GITHUB_PKG_DIR), "github_package_size_mb": _dir_size_mb(GITHUB_PKG_DIR),
    "kaggle_package_files": _dir_file_count(KAGGLE_PKG_DIR), "kaggle_package_size_mb": _dir_size_mb(KAGGLE_PKG_DIR),
    "linkedin_package_files": _dir_file_count(LINKEDIN_PKG_DIR),
    "code_quality_files_checked": _n_checked, "code_quality_files_valid": _n_valid,
    "packaging_manifest_included": _n_included, "packaging_manifest_excluded": _n_excluded,
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb18_summary_path = ARTIFACTS_DIR / "notebook_18_summary.json"
with open(nb18_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_18_summary, f, indent=2)
print(f"\u2705 Saved -> {nb18_summary_path}")
print("\n\u2705 Section 18 complete.")


# =============================================================================
# SECTION 19: FINAL COMPLETION SUMMARY -- ALL 18 NOTEBOOKS
# =============================================================================
_section("SECTION 19: Notebook 18 Complete -- Platform Build Finished")

print("NOTEBOOK 18: REPOSITORY PACKAGING -- COMPLETE")
print(f"  GitHub package    : {_dir_file_count(GITHUB_PKG_DIR)} files, {_dir_size_mb(GITHUB_PKG_DIR)} MB -> {GITHUB_PKG_DIR}")
print(f"  Kaggle package     : {_dir_file_count(KAGGLE_PKG_DIR)} files, {_dir_size_mb(KAGGLE_PKG_DIR)} MB -> {KAGGLE_PKG_DIR}")
print(f"  LinkedIn showcase  : {_dir_file_count(LINKEDIN_PKG_DIR)} files -> {LINKEDIN_PKG_DIR}")
print(f"  Code quality       : {_n_valid} / {_n_checked} files syntax-valid")
print(f"  Files produced      : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb18_summary_path]:
    print(f"    - {_p.name}")
print(f"\n  This is Notebook 18 of 18 -- the AMEX Enterprise Credit Risk Platform build is complete.")
print(f"  Re-run Notebook 17 (Comprehensive Reporting) any time to see the platform's live overall status.")
print("\n\u2705 Ready to share.")
